# Serving tier on a Colab T4: engine, parity gate, latency sweep

Runs `docs/AGENT_BRIEF.md` Phase 4. Two processes share one card: the LLM
serving engine, and the pipeline that talks to it over HTTP.

**Runtime → Change runtime type → T4 GPU** before starting.

Order matters and is not arbitrary:

1. install both repos
2. **patch the engine's streaming loop** — it corrupts Indic text, and this
   must happen before the server starts, because uvicorn imports the module
   once
3. start the engine on **Qwen3-1.7B** with a pool sized for one stream, and
   **wait for `/ready`** — not `/health`, which answers alive all through
   CUDA-graph capture and Triton JIT
4. `scripts/engine_parity.py` — the gate. Read `corrupted_prompts` before the
   agreement figures; while it is non-zero the agreement rate measures
   corruption, not whether the two decoders agree
5. `scripts/latency_ab.py` — four interleaved arms
6. apply the rules already written in `docs/EXPERIMENTS.md`, without
   re-deriving them from the numbers you just saw

Of a 4435.5 ms p50 turn, the LLM was 3324 ms and ASR was 342 ms *and off the
critical path*. This notebook is aimed at the 3324 ms.

The model is **Qwen3-1.7B**, served by the engine, with an explicit-1.7B arm
as the baseline so the sweep isolates the engine rather than confounding it
with a model change.

Why not 4B: the parity gate decodes the same weights **twice** on one card —
the server holds a copy and the gate needs one in its own process. Two 4B
copies is ~15 GiB of weights on a 15 GiB T4, and the second load dies partway
through. 1.7B fits twice over with room for Whisper, and on this project's own
T4 bake-off it keeps Devanagari ratio **1.000** where 4B drops to 0.988, at
2534 ms first-sentence p50 against 4B's 3553 ms.

The 0.6B-versus-1.7B question is already answered by that bake-off and is not
what this notebook measures.

## 1 · GPU and both repositories

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
# The pipeline. Use your own remote if you have push access.
%cd /content
!git clone -q https://github.com/Vaibhav7711/indic-voice-pipeline.git || true
%cd /content/indic-voice-pipeline
!git pull -q --ff-only || true
!pip install -q -e ".[audio]" 2>&1 | tail -2
!git log --oneline -1

In [ ]:
# The serving engine, beside it. Its extras: torch/transformers/triton must
# match the runtime's CUDA, which on Colab they already do.
%cd /content
!git clone -q https://github.com/Vaibhav7711/full-inference-engine.git || true
%cd /content/full-inference-engine
!git pull -q --ff-only || true
!pip install -q -e ".[server]" 2>&1 | tail -3
!git log --oneline -1

### What this device will actually serve with

`check_hooks.py --backends-only` prints the checkpoint's geometry, what got
fused, and every attention backend with `ok`/`no` **and the reason**. On a T4
(sm_75) expect FlashAttention to be unavailable — its wheels need sm_80+ —
and the tiled Triton prefill to be refused because `tl.dot` emits no
`mma.sync` there. Both are measured negative results, not failures. The
engine's per-architecture policy picks the settings an A/B on *this*
architecture chose.

In [ ]:
%cd /content/full-inference-engine
!python scripts/check_hooks.py --backends-only

## 2 · Patch the engine's streaming loop

**Required, and it must happen before the server starts** — uvicorn imports
the module once.

A measured T4 run found the engine corrupts streamed Indic text: 1/1 English
prompt identical, 0/4 Hindi prompts agreeing, every Hindi response studded
with U+FFFD. `नमस्ते, मौसम कैसा है?` came back as `नमस्�े, म�सम क�सा ह�?`,
with the correct characters *dropped*.

Qwen's byte-level BPE splits a 3-byte Devanagari character across two tokens.
`_stream` decodes the tokens received so far and sends
`decoded[len(already_sent):]` — a slice by length. While a character is
half-arrived that decode ends in U+FFFD, which gets sent; when the rest
arrives the corrected text is no longer an extension of what was sent, so the
replacement character can never be retracted and the real character is
skipped. ASCII never triggers it, which is why English passes.

The patch holds back a trailing U+FFFD and stops assuming the prefix: 14
lines, in [`docs/ENGINE_BUG_UTF8_STREAMING.md`](https://github.com/Vaibhav7711/indic-voice-pipeline/blob/main/docs/ENGINE_BUG_UTF8_STREAMING.md).
The patcher is idempotent, so re-running this notebook is safe, and it refuses
rather than guessing if the engine has since fixed this itself.

In [ ]:
%cd /content/indic-voice-pipeline
!python scripts/patch_engine_utf8.py --engine-root /content/full-inference-engine
# Expect "patched:" on a fresh clone, or "already-patched:" if this cell
# has run before. "not-found" means the engine's _stream has changed --
# read the script's message before forcing anything.
!python scripts/patch_engine_utf8.py --engine-root /content/full-inference-engine --check
!git -C /content/full-inference-engine diff --stat

Prove the fix on the actual tokenizer before spending GPU time on it.
This is CPU-only and takes seconds: it streams a Devanagari string token by
token, the old way and the patched way.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B')
FFFD = chr(0xFFFD)
text = 'नमस्ते, मौसम कैसा है?'
ids = tok(text, add_special_tokens=False).input_ids

def stream(ids, holdback):
    sent, out = '', []
    for n in range(1, len(ids) + 1):
        decoded = tok.decode(ids[:n], skip_special_tokens=True)
        if holdback:
            stable = decoded[:-1] if decoded.endswith(FFFD) else decoded
            if stable.startswith(sent) and len(stable) > len(sent):
                out.append(stable[len(sent):]); sent = stable
        else:
            if len(decoded) > len(sent):
                out.append(decoded[len(sent):]); sent = decoded
    return ''.join(out)

before, after = stream(ids, False), stream(ids, True)
print('tokens        :', len(ids))
print('expected      :', text)
print('old  _stream  :', before, '   U+FFFD:', before.count(FFFD))
print('patched       :', after, '   U+FFFD:', after.count(FFFD))
assert after == text, 'the hold-back did not reproduce the text'
print('\nfix verified on this checkpoint')

## 3 · Start the engine and wait for warmup

`nohup` so the cell returns; `--wait-only` then blocks until `/ready` is 200.
Warmup is a startup cost — graph capture across every forward shape plus
Triton JIT — and it must not land inside the first measured turn.

If `--wait-only` times out or reports the process gone, read `engine.log`:
the engine refuses geometry its paged kernels cannot serve *at load, with the
reason*, and that message is the answer.

In [ ]:
import os, signal, subprocess, sys, threading, time
os.chdir('/content/indic-voice-pipeline')
sys.path.insert(0, '/content/indic-voice-pipeline')

# Qwen3-1.7B through this repo's factory, which sizes the pool for one
# stream: 512 x 16 = 8192 KV tokens at 112 KiB/token = 0.875 GiB, against
# 1.75 GiB for the engine's 1024-block default. Weights are ~3.8 GiB.
#
# Why 1.7B and not 4B: the parity gate decodes the same weights twice on one
# card -- the server holds a copy and the gate needs one too. Two 4B copies is
# ~15 GiB of weights on a 15 GiB T4 and dies partway through the second load.
# 1.7B fits twice over with room for Whisper, and the T4 bake-off has it at
# Devanagari ratio 1.000 against 4B's 0.988.
env = dict(os.environ,
           # The engine comes first so its own imports resolve to its own
           # tree. Five top-level names collide between the two repos
           # (benchmarks, docs, results, scripts, tests), and the engine's
           # scripts/ is a real package while this repo's is a namespace
           # package, so anything under scripts. here would be shadowed.
           # Hence the factory lives at llm.engines.server_app.
           PYTHONPATH='/content/full-inference-engine:/content/indic-voice-pipeline',
           LLM_SERVER_MODEL='Qwen/Qwen3-1.7B',
           LLM_SERVER_NUM_BLOCKS='512',
           LLM_SERVER_BLOCK_SIZE='16',
           LLM_SERVER_MAX_ACTIVE='2',
           LLM_SERVER_GRAPH_BUCKETS='1,2',
           LLM_SERVER_DTYPE='float16')   # must match engine_parity --dtype

# An HF token lifts the unauthenticated rate limit. Qwen3-1.7B is ~4 GB, and a
# throttled download is the most common reason this cell takes a long time.
# Set it in Colab's Secrets (key icon) as HF_TOKEN; read access is enough.
try:
    from google.colab import userdata
    env['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('using the HF_TOKEN secret')
except Exception as error:
    print(f'no HF_TOKEN secret ({type(error).__name__}); downloading '
          f'unauthenticated, which may be throttled')

from llm.engines.server_app import describe, resolve_config
from scripts.serve_llm import wait_until_ready
print('memory before allocating:', describe(resolve_config(env)))

LOG = '/content/engine.log'
log = open(LOG, 'w')
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'llm.engines.server_app:create', '--factory',
     '--host', '127.0.0.1', '--port', '8000'],
    stdout=log, stderr=subprocess.STDOUT, env=env)
print('pid', server.pid, '- log:', LOG)

# The server's own output, live. Without this, an 8 GB download and a crash
# look identical from outside: both are "connection refused".
def tail():
    with open(LOG) as handle:
        while server.poll() is None:
            line = handle.readline()
            if line:
                print('  |', line.rstrip(), flush=True)
            else:
                time.sleep(0.5)

threading.Thread(target=tail, daemon=True).start()

# `process=server` is the point: wait_until_ready raises the moment the server
# exits, instead of polling a corpse until the deadline. Warmup is graph
# capture plus Triton JIT on top of the download, hence the long timeout.
try:
    body = wait_until_ready('http://127.0.0.1:8000', timeout_s=2400,
                            process=server)
    print('\nREADY:', body)
except RuntimeError as error:
    print(f'\nFAILED: {error}\n')
    print('--- last 40 lines of engine.log ---')
    print(''.join(open(LOG).readlines()[-40:]))
    raise

#### If that failed

The traceback is in `/content/engine.log` and printed above. The three
usual causes:

- **`ModuleNotFoundError: engine`** — the engine's `pip install -e ".[server]"`
  in step 1 did not finish. Re-run it and read its output rather than the
  last three lines.
- **`torch.OutOfMemoryError`** — something else holds VRAM. `nvidia-smi` will
  say what. Lower `LLM_SERVER_NUM_BLOCKS` to `256`, or serve Qwen3-0.6B.
- **the engine refusing the checkpoint's geometry, with the reason** — that is
  the engine working as designed; the message names what its paged kernels
  cannot serve.

A long wait with no output at all is usually the 8 GB download being
throttled, not a hang. The cell below shows where it actually is.

In [ ]:
# Where it actually got to.
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
!ls -la ~/.cache/huggingface/hub 2>/dev/null | tail -5
print('--- engine.log ---')
!tail -40 /content/engine.log
print('--- is anything listening on 8000? ---')
!curl -s -m 3 localhost:8000/health || echo "nothing listening"

In [ ]:
# What the engine says about itself once it is up.
!curl -s localhost:8000/v1/models && echo
!curl -s localhost:8000/ready && echo

## 4 · The parity gate

Both decoders are greedy: the engine documents `temperature = 0` as greedy
and taking precedence over every other knob, and the explicit runner is
greedy. Same weights, same prompt, same text — or the faster engine is not
serving the same model.

Gates on the first 24 characters, which is what a listener hears before the
first unit reaches TTS. Late divergence is fp16 non-associativity and passes
by design; **early** divergence means prefill differs and is a real defect.

Exit status 0 or the rest of this notebook does not count.

In [ ]:
# --dtype must match what the server loaded. A bf16 reference against an fp16
# server is two different sets of numerics, and two greedy decoders over
# different numerics diverge for reasons that say nothing about the engine.
#
# This loads a SECOND copy of the weights: the server holds one, and the
# reference runner needs one here. The script's vram preflight checks that fits
# before spending two minutes on the load.
import subprocess
completed = subprocess.run(
    ['python', 'scripts/engine_parity.py',
     '--llm-model', 'Qwen/Qwen3-1.7B',
     '--dtype', 'float16',
     '--llm-base-url', 'http://127.0.0.1:8000/v1',
     '--note', 'colab t4, qwen3-1.7b, llm.engines.server_app 512x16'])
print('exit status:', completed.returncode,
      '(0 = parity holds, 1 = it does not, 3 = vram preflight refused)')

In [ ]:
import json, os
PATH = 'results/engine_parity/parity.json'
if not os.path.exists(PATH):
    # The gate did not get far enough to write a report. Its own output above
    # is the error; do not read a missing file and report that instead.
    raise SystemExit('no report written -- read the gate output above. exit 3 '
                     'is the vram preflight refusing; a killed process means '
                     'the second copy of the weights did not fit.')

r = json.load(open(PATH))
print(json.dumps(r['summary'], indent=2))
print('vram preflight:', r['vram_preflight'])

# Read corrupted_prompts FIRST. While it is non-zero the agreement figures
# measure the corruption, not decoder agreement: Qwen's byte-level BPE splits
# a 3-byte Devanagari character across two tokens, and a server that streams
# the diff of its decoded-so-far text emits U+FFFD for the partial character
# and cannot retract it. See docs/ENGINE_BUG_UTF8_STREAMING.md.
if r['summary']['corrupted_prompts']:
    print(f"\nSTOP: {r['summary']['corrupted_prompts']} corrupted responses, "
          f"{r['summary']['served_replacement_chars']} U+FFFD characters.")
    print('Step 2 did not take effect. Re-run it, then restart the server.')
for item in r['results']:
    if item['comparison']['corruption']:
        print('CORRUPT:', item['prompt']); print('  srv:', item['served'][:120])
    elif not item['comparison']['agreed']:
        print('DIVERGED:', item['prompt']); print('  ref:', item['reference'][:120])
        print('  srv:', item['served'][:120])

## 5 · The latency sweep

Four arms, interleaved one turn each per round, sharing one set of loaded
weights. `--rounds 8` because the history-budget effect grows with session
length — at turn 1 there is no history to trim, so a short run understates
it.

`--tts mms` keeps a network voice's bad minute out of the measurement. Drop
to `--tts edge` only if VRAM is tight.

In [ ]:
# Baseline and served are the SAME checkpoint, so the only difference between
# them is the engine. Mixing a model change into this comparison would make it
# unattributable -- the 0.6B-vs-1.7B question is already answered by the T4
# bake-off (1744 ms vs 2534 ms first-sentence p50 on the explicit runner).
#
# Both a local 1.7B (~3.8 GiB) and the server's copy are resident, the same
# two-copy cost the parity gate has.
!python scripts/latency_ab.py --rounds 8 \
    --llm-engine explicit --llm-engine http \
    --llm-model Qwen/Qwen3-1.7B \
    --llm-base-url http://127.0.0.1:8000/v1 \
    --tts mms \
    --arm baseline \
    --arm served:llm_engine=http \
    --arm history200:history_tokens=200 \
    --arm units30:unit_chars=30 \
    --note "colab t4; baseline=explicit 1.7B, served=engine 1.7B"

In [ ]:
import json
summary = json.load(open('results/latency_ab/summary.json'))
PRIMARY = summary['primary_metric']       # committed transcript -> first audio
print('primary metric:', PRIMARY)

def fmt(v):
    return f'{v:8.1f}' if isinstance(v, (int, float)) else '     n/a'

for name, block in summary['arms'].items():
    print(f"{name:<14} n={block['turns']:<3} "
          f"p50={fmt(block[PRIMARY]['p50'])}  "
          f"p90={fmt(block[PRIMARY]['p90'])}  "
          f"first_token={fmt(block['final_transcript_to_first_llm_token_ms']['p50'])}  "
          f"to_unit={fmt(block['first_token_to_first_unit_ms']['p50'])}")

# response_latency_ms is n/a here by construction: these turns carry no speech,
# so the endpoint-to-final segment does not exist and is not made up. The
# measured 660.8 ms floor sits under any perceived-latency figure.
base = summary['arms']['baseline'][PRIMARY]['p50']
served = summary['arms']['served'][PRIMARY]['p50']
if isinstance(base, (int, float)) and isinstance(served, (int, float)) and served:
    print(f"\nengine vs explicit, same checkpoint: {base / served:.4f}x "
          f"on {PRIMARY} p50")
    print("pre-registered rule: adopt the served engine above 1.1x, and only if")
    print("the parity gate passed. Below 1.1x keep the explicit runner and")
    print("record the measured ratio.")
else:
    print("\nno ratio: an arm did not reach audio. That is a failure to "
          "investigate, not a missing number to fill in.")

## 6 · The human conditions

Two arms cannot be closed by a number, and the pre-registered rules say so.

**`history200`** — does a 200-token budget still hold a conversation? Ask a
follow-up that depends on the previous turn. If the assistant answers as
though the exchange never happened, the latency it bought is not a win for a
dialogue agent.

**`units30`** — listen. A shorter first unit buys silence-to-speech by
cutting the sentence in a slightly odder place.

In [ ]:
# Text in, audio out: this checks dialogue memory, not ASR, so feeding fixed
# text is what makes the two history budgets comparable at all. Same builder
# the A/B uses, so the arm here is the arm that was measured.
import os, sys
sys.path.insert(0, '/content/indic-voice-pipeline')
os.chdir('/content/indic-voice-pipeline')

from agent.audio import DecodingBufferSink
from llm.engines import build_llm
from scripts.latency_ab import build_arm
from tts.local import MmsTtsSynthesizer

generator, tokenizer, info = build_llm(
    'http', model='Qwen/Qwen3-1.7B', base_url='http://127.0.0.1:8000/v1')
print(info)
synth = MmsTtsSynthesizer('hi'); synth.warm_up()

# A follow-up whose meaning depends on the previous turn. If the shorter
# budget has dropped the antecedent, the second answer will not resolve
# "वहाँ" and the third will not resolve "उसका".
FOLLOW_UPS = ["भारत की राजधानी क्या है?",
              "वहाँ का मौसम कैसा रहता है?",
              "उसका मतलब क्या है?"]

for tokens in (800, 200):
    turn = build_arm({'history_tokens': tokens},
                     base={'history_tokens': 800, 'history_turns': 6,
                           'unit_chars': 60, 'llm_engine': 'http'},
                     generators={'http': (generator, tokenizer)}, synth=synth,
                     sink_factory=lambda: DecodingBufferSink(synth.format))
    print(f"\n=== max_history_tokens={tokens} ===")
    for question in FOLLOW_UPS:
        result = turn.run(question)
        latency = result.as_dict().get('response_latency_ms')
        shown = f"{latency:.0f} ms" if isinstance(latency, (int, float)) else "n/a"
        print(f"  Q: {question}\n  A: {result.response}  [{shown}]")

print("\nRead the 200-token answers: did the follow-ups still resolve? That "
      "judgement is yours, not a threshold's.")

## 7 · Record it

Write the numbers into the ledger under the pre-registered section, apply the
rules literally, and commit. An arm that was measured but not decided, or
decided but not recorded, is the same as one that never ran.

In [ ]:
!git -C /content/indic-voice-pipeline add results/engine_parity results/latency_ab
!git -C /content/indic-voice-pipeline status --short

In [ ]:
# Stop the engine. SIGINT drains in-flight requests rather than killing them.
import signal
server.send_signal(signal.SIGINT)
print('exit', server.wait(timeout=60))
!tail -3 /content/engine.log